# NFL EPA Model - Data Exploration

**Phase 2a: EPA Model Development**

This notebook explores the nflfastR play-by-play data (2016-2024) to understand:
1. Data structure and key columns
2. Expected Points (EP) distribution by field position
3. Down and distance patterns
4. Identifying blowout games for filtering
5. Data quality issues

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Data

In [ ]:
# Load combined data
print("Loading data...")
pbp = pd.read_parquet('../data/play_by_play_2016_2024.parquet')

print(f"✓ Loaded {len(pbp):,} plays")
print(f"  Seasons: {sorted(pbp['season'].unique().tolist())}")
print(f"  Columns: {len(pbp.columns)}")
print(f"  Memory: {pbp.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 2. Understand Data Structure

In [ ]:
# Display basic info
print("Dataset shape:", pbp.shape)
print("\nFirst few rows:")
pbp.head()

In [ ]:
# Key columns for EPA model
epa_columns = [
    'play_id', 'game_id', 'season', 'week',
    'posteam', 'defteam', 'home_team', 'away_team',
    'down', 'ydstogo', 'yardline_100',
    'quarter', 'quarter_seconds_remaining', 'game_seconds_remaining',
    'posteam_score', 'defteam_score', 'score_differential',
    'ep', 'epa', 'wp', 'vegas_wp',
    'play_type', 'touchdown', 'fumble', 'interception',
    'penalty', 'qb_kneel', 'qb_spike'
]

print("Key columns for EPA model:")
print("\n".join(epa_columns))

# Check if all columns exist
missing_cols = [col for col in epa_columns if col not in pbp.columns]
if missing_cols:
    print(f"\n⚠️  Missing columns: {missing_cols}")
else:
    print("\n✓ All key columns present")

In [ ]:
# Check for missing values in key columns
key_cols = ['down', 'ydstogo', 'yardline_100', 'ep', 'wp']
print("Missing values in key columns:")
print(pbp[key_cols].isnull().sum())
print(f"\nTotal plays: {len(pbp):,}")
print(f"Plays with EP: {pbp['ep'].notna().sum():,}")
print(f"Plays with WP: {pbp['wp'].notna().sum():,}")

## 3. Explore Expected Points Distribution

In [ ]:
# Filter to plays with EP values
pbp_with_ep = pbp[pbp['ep'].notna()].copy()
print(f"Plays with EP values: {len(pbp_with_ep):,} ({len(pbp_with_ep)/len(pbp)*100:.1f}%)")

# EP distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(pbp_with_ep['ep'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Expected Points (EP)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Expected Points')
axes[0].axvline(pbp_with_ep['ep'].median(), color='red', linestyle='--', label=f'Median: {pbp_with_ep["ep"].median():.2f}')
axes[0].legend()

# Box plot
axes[1].boxplot(pbp_with_ep['ep'], vert=True)
axes[1].set_ylabel('Expected Points (EP)')
axes[1].set_title('EP Box Plot')

plt.tight_layout()
plt.show()

print(f"\nEP Statistics:")
print(pbp_with_ep['ep'].describe())

## 4. EP by Field Position

In [ ]:
# Group by field position (yardline_100)
ep_by_position = pbp_with_ep.groupby('yardline_100')['ep'].agg(['mean', 'median', 'count']).reset_index()

# Plot EP by field position
plt.figure(figsize=(14, 6))
plt.plot(ep_by_position['yardline_100'], ep_by_position['mean'], label='Mean EP', linewidth=2)
plt.plot(ep_by_position['yardline_100'], ep_by_position['median'], label='Median EP', linewidth=2, alpha=0.7)
plt.axvline(20, color='red', linestyle='--', alpha=0.5, label='Red Zone (20 yards)')
plt.axvline(50, color='green', linestyle='--', alpha=0.5, label='Midfield')
plt.xlabel('Yards to Goal')
plt.ylabel('Expected Points')
plt.title('Expected Points by Field Position')
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().invert_xaxis()  # Invert x-axis so goal line is on the right
plt.show()

print("Key observations:")
print(f"  EP at own 1-yard line: {ep_by_position[ep_by_position['yardline_100']==99]['mean'].values[0]:.2f}")
print(f"  EP at midfield: {ep_by_position[ep_by_position['yardline_100']==50]['mean'].values[0]:.2f}")
print(f"  EP at opponent 20 (red zone): {ep_by_position[ep_by_position['yardline_100']==20]['mean'].values[0]:.2f}")
print(f"  EP at opponent 1-yard line: {ep_by_position[ep_by_position['yardline_100']==1]['mean'].values[0]:.2f}")

## 5. EP by Down and Distance

In [ ]:
# Filter to valid downs (1-4)
pbp_valid_down = pbp_with_ep[pbp_with_ep['down'].isin([1, 2, 3, 4])].copy()

# EP by down
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot by down
pbp_valid_down.boxplot(column='ep', by='down', ax=axes[0])
axes[0].set_xlabel('Down')
axes[0].set_ylabel('Expected Points')
axes[0].set_title('EP Distribution by Down')
plt.sca(axes[0])
plt.xticks([1, 2, 3, 4], ['1st', '2nd', '3rd', '4th'])

# Mean EP by down
ep_by_down = pbp_valid_down.groupby('down')['ep'].mean()
axes[1].bar(ep_by_down.index, ep_by_down.values, color=['green', 'blue', 'orange', 'red'])
axes[1].set_xlabel('Down')
axes[1].set_ylabel('Mean Expected Points')
axes[1].set_title('Mean EP by Down')
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_xticklabels(['1st', '2nd', '3rd', '4th'])

plt.tight_layout()
plt.show()

print("\nMean EP by down:")
print(ep_by_down)

## 6. Next Steps

**Completed:**
- ✅ Loaded 435,483 plays from 2016-2024
- ✅ Explored EP distribution
- ✅ Analyzed EP by field position
- ✅ Examined EP by down

**TODO (Next Notebooks):**
1. Identify and filter blowout games
2. Data cleaning (remove penalties, kneels, spikes, etc.)
3. Feature engineering (red zone flag, home field advantage)
4. Build XGBoost EPA model
5. Implement LOSO cross-validation
6. Evaluate model (calibration error, MAE, RMSE, R²)

In [ ]:
# Save summary statistics for reference
summary = {
    'total_plays': len(pbp),
    'plays_with_ep': len(pbp_with_ep),
    'seasons': sorted(pbp['season'].unique().tolist()),
    'ep_mean': pbp_with_ep['ep'].mean(),
    'ep_std': pbp_with_ep['ep'].std(),
    'ep_min': pbp_with_ep['ep'].min(),
    'ep_max': pbp_with_ep['ep'].max()
}

print("Summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")